# Large Dataset Handling & Chunked Processing

## Definition
When datasets are too large to fit in RAM, you need strategies to process them efficiently
without loading everything at once.

### Strategies for Large Data
| Strategy | When to Use |
|----------|-------------|
| **Column selection at read** | Load only needed columns |
| **dtype optimization at read** | Reduce memory before loading |
| **Chunked reading** | File larger than RAM |
| **Filter at read time** | Skip irrelevant rows early |
| **Parquet format** | Binary, columnar, compressed |
| **Dask** | Parallel, out-of-core pandas |

### Memory-Usage Formula
```
DataFrame memory = n_rows x n_cols x bytes_per_dtype

Example:
  10M rows x 20 float64 cols = 1.6 GB
  10M rows x 20 float32 cols = 800 MB  (50% saving)
```

In [ ]:
import pandas as pd
import numpy as np
import os
import time

rng = np.random.default_rng(42)
n = 200_000

# Generate a "large" CSV for demo
df_large = pd.DataFrame({
    'id':       range(n),
    'age':      rng.integers(18, 65, n),
    'income':   rng.normal(60000, 20000, n),
    'score':    rng.uniform(0, 100, n),
    'category': rng.choice(['A', 'B', 'C', 'D'], n),
    'target':   rng.choice([0, 1], n, p=[0.8, 0.2])
})
csv_path = '/tmp/large_data.csv'
df_large.to_csv(csv_path, index=False)
file_size = os.path.getsize(csv_path) / 1024**2
print(f'CSV size: {file_size:.1f} MB  |  Rows: {n:,}')

In [ ]:
# Strategy 1: Load only needed columns
t0 = time.time()
df_subset = pd.read_csv(csv_path, usecols=['id', 'income', 'target'])
print(f'Subset load time: {(time.time()-t0)*1000:.0f}ms')
print(f'Memory: {df_subset.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print('Shape:', df_subset.shape)

In [ ]:
# Strategy 2: Specify dtypes at read time (massive memory saving)
dtypes = {
    'id':       'int32',
    'age':      'int8',
    'income':   'float32',
    'score':    'float32',
    'category': 'category',
    'target':   'int8'
}

t0 = time.time()
df_typed = pd.read_csv(csv_path, dtype=dtypes)
print(f'Typed load time: {(time.time()-t0)*1000:.0f}ms')
print(f'Memory (default dtypes): {df_large.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print(f'Memory (optimized dtypes): {df_typed.memory_usage(deep=True).sum()/1024**2:.2f} MB')

In [ ]:
# Strategy 3: Chunked reading for aggregation across a huge file
chunk_size = 10_000
category_stats = {}

for chunk in pd.read_csv(csv_path, chunksize=chunk_size, dtype=dtypes):
    for cat, grp in chunk.groupby('category'):
        if cat not in category_stats:
            category_stats[cat] = {'sum': 0.0, 'count': 0}
        category_stats[cat]['sum']   += grp['income'].sum()
        category_stats[cat]['count'] += len(grp)

result = {cat: vals['sum'] / vals['count']
          for cat, vals in category_stats.items()}

print('Category mean income (chunked processing):')
for k, v in sorted(result.items()):
    print(f'  {k}: {v:,.0f}')

In [ ]:
# Strategy 4: Filter rows during chunked read (load only positive class)
positives = []
for chunk in pd.read_csv(csv_path, chunksize=10_000, dtype=dtypes):
    positives.append(chunk[chunk['target'] == 1])

df_positives = pd.concat(positives, ignore_index=True)
print(f'Positive class rows: {len(df_positives):,}  (expected ~{int(n*0.2):,})')

In [ ]:
# Strategy 5: Chunked multi-column aggregation
agg_results = []
for chunk in pd.read_csv(csv_path, chunksize=20_000, dtype=dtypes):
    agg = chunk.groupby('category').agg(
        income_sum=('income', 'sum'),
        income_count=('income', 'count'),
        score_sum=('score', 'sum')
    )
    agg_results.append(agg)

combined = pd.concat(agg_results).groupby(level=0).sum()
combined['income_mean'] = combined['income_sum'] / combined['income_count']
combined['score_mean']  = combined['score_sum']  / combined['income_count']
print('Chunked aggregation result:')
print(combined[['income_mean', 'score_mean']].round(2))

In [ ]:
# Strategy 6: Dask for parallel out-of-core processing
try:
    import dask.dataframe as dd
    ddf = dd.read_csv(csv_path, dtype={k: str(v) for k, v in dtypes.items()
                                        if k not in ['category']})
    print('Dask DataFrame (lazy):')
    print(ddf.dtypes)
    result = ddf.groupby('category')['income'].mean().compute()
    print('\nDask category means:')
    print(result.round(2))
except ImportError:
    print('Dask not installed: pip install dask[dataframe]')
    print('Chunked pandas approach above works as an alternative.')
finally:
    os.unlink(csv_path)
    print('Cleaned up CSV.')